# **Load previously-trained GPT2-MoE model from Google Drive and evaluate**

Import

In [3]:
import torch
import copy
import torch.nn as nn
from transformers import AutoTokenizer, GPT2LMHeadModel

from collections import Counter

from google.colab import drive
import os

from safetensors.torch import load_file

Config

In [4]:
# environment setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_path = "/content/drive/MyDrive/gpt2-moe-alpaca-2/checkpoint-1120"

Mount Google Drive

In [5]:
drive.mount('/content/drive')

Mounted at /content/drive


Architecture

In [6]:
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F


class ChunkLinearMoE(nn.Module):
  """A single Linear layer split into 'num_chunks' chunks, where each chunk has

  'num_options' independent candidate sub-matrices.
  """

  def __init__(self, in_features, out_features, num_chunks=10, num_options=4):
    super().__init__()
    self.in_features = in_features
    self.out_features = out_features
    self.num_chunks = num_chunks
    self.num_options = num_options

    assert (
        out_features % num_chunks == 0
    ), f"out_features ({out_features}) must be divisible by num_chunks ({num_chunks})"
    self.chunk_dim = out_features // num_chunks

    # Router: Projects input to (num_chunks * num_options) logits
    self.router = nn.Linear(in_features, num_chunks * num_options)

    # Experts: Parameter bank of shape [num_chunks, num_options, chunk_dim, in_features]
    # This stores the weights for all 4 options across all 10 chunks efficiently!
    self.chunk_weights = nn.Parameter(
        torch.randn(num_chunks, num_options, self.chunk_dim, in_features)
        * 0.02
    )
    self.chunk_biases = nn.Parameter(
        torch.zeros(num_chunks, num_options, self.chunk_dim)
    )

  def forward(self, x):
    # x shape: [batch_size * seq_len, in_features]
    N, D_in = x.shape

    # 1. Compute Router Logits and Probs
    router_logits = self.router(x).view(N, self.num_chunks, self.num_options)
    router_probs = F.softmax(router_logits, dim=-1)  # Softmax over num_options

    # 2. Top-1 selection per chunk (non-vectorized for loops)
    outputs = []
    for i in range(self.num_chunks):
      # Get top 1 expert for this chunk
      chunk_router_probs = router_probs[:, i, :]
      top1_weights, top1_indices = torch.topk(chunk_router_probs, k=1, dim=-1)
      expert_idx = top1_indices.squeeze(-1)  # Shape: [N]

      # Gather selected weights and biases for this chunk
      selected_w = self.chunk_weights[i, expert_idx]  # Shape: [N, chunk_dim, in_features]
      selected_b = self.chunk_biases[i, expert_idx]    # Shape: [N, chunk_dim]

      # Perform linear transformation for this chunk
      chunk_output = torch.bmm(x.unsqueeze(1), selected_w.transpose(1, 2)).squeeze(1) + selected_b # Shape: [N, chunk_dim]

      # Apply router confidence scores
      chunk_output = chunk_output * top1_weights
      outputs.append(chunk_output)

    # 3. Concatenate all chunk outputs
    return torch.cat(outputs, dim=-1)  # Shape: [N, out_features]


class ChunkMoELayer(nn.Module):
  """Complete Bi-layer MLP replacing standard GPT-2 MLP with Chunk-Level MoE."""

  def __init__(
      self, hidden_dim=768, intermediate_dim=3072, num_chunks=10, num_options=4
  ):
    super().__init__()
    # Layer 1: Up-projection (768 -> 3072, split into 10 chunks)
    self.c_fc = ChunkLinearMoE(
        hidden_dim, intermediate_dim, num_chunks, num_options
    )

    # Activation Function
    self.act = nn.GELU()

    # Layer 2: Down-projection (3072 -> 768, split into 10 chunks)
    self.c_proj = ChunkLinearMoE(
        intermediate_dim, hidden_dim, num_chunks, num_options
    )

  def forward(self, hidden_states):
    orig_shape = hidden_states.shape
    flat_x = hidden_states.view(-1, orig_shape[-1])

    # Bi-layer execution with chunk routing
    h1 = self.act(self.c_fc(flat_x))
    out = self.c_proj(h1)

    return out.view(orig_shape)

In [ ]:
# define GPT2-MoE architecture (with routing logger)

class GPT2MoELayer(nn.Module):
    def __init__(self, original_mlp, num_experts=4):
        super().__init__()
        self.num_experts = num_experts

        # Extract hidden dimension
        hidden_dim = original_mlp.c_fc.weight.shape[0]

        # Router network and expert clones
        self.router = nn.Linear(hidden_dim, num_experts)
        self.experts = nn.ModuleList([copy.deepcopy(original_mlp) for _ in range(num_experts)])

        # Tracking telemetry (safe for training, defaults to off)
        self.routing_log = []
        self.track_routing = False

    def forward(self, hidden_states):
        orig_shape = hidden_states.shape
        flat_hidden_states = hidden_states.view(-1, orig_shape[-1])

        # Compute routing probabilities
        router_logits = self.router(flat_hidden_states)
        router_probs = torch.softmax(router_logits, dim=-1)

        # Select top-1 expert
        top1_weights, top1_indices = torch.topk(router_probs, k=1, dim=-1)

        # Log decisions if telemetry is explicitly turned on
        if self.track_routing:
            chosen_experts = top1_indices.squeeze(-1).tolist()
            if isinstance(chosen_experts, list):
                self.routing_log.extend(chosen_experts)
            else:
                self.routing_log.append(chosen_experts)

        # Route tokens to their designated expert
        flat_outputs = torch.zeros_like(flat_hidden_states)
        for expert_idx in range(self.num_experts):
            mask = (top1_indices.squeeze(-1) == expert_idx)

            if mask.any():
                expert_inputs = flat_hidden_states[mask]
                expert_outputs = self.experts[expert_idx](expert_inputs)
                flat_outputs[mask] = expert_outputs * top1_weights[mask]

        return flat_outputs.view(orig_shape)

In [7]:
tokenizer = AutoTokenizer.from_pretrained('openai-community/gpt2')
model = GPT2LMHeadModel.from_pretrained('openai-community/gpt2')

# Sync pad token configuration
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

# revise baseline architecture to MoE
for i in range(len(model.transformer.h)):
    original_mlp = model.transformer.h[i].mlp
    model.transformer.h[i].mlp = ChunkMoELayer(hidden_dim=768, intermediate_dim=2992, num_chunks=8, num_options=4)

# apply fine-tuned MoE weights from Google Drive
state_dict = load_file(f"{model_path}/model.safetensors")
model.load_state_dict(state_dict, strict=False) # set to strict=False because we don't have the same weights with the new MoE architecture
model.to(device)
model.eval()

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ChunkMoELayer(
          (c_fc): ChunkLinearMoE(
            (router): Linear(in_features=768, out_features=32, bias=True)
          )
          (act): GELU(approximate='none')
          (c_proj): ChunkLinearMoE(
            (router): Linear(in_features=2992, out_features=32, bias=True)
          )
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_h

Evaluation

In [ ]:
# evaluation function with routing logging

def evaluate_phrase_with_routing(instruction, input_context=""):
    if input_context:
        prompt = (
            f"Below is an instruction that describes a task, paired with an input that provides further context. "
            f"Write a response that appropriately completes the request.\n\n"
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{input_context}\n\n"
            f"### Response:\n"
        )
    else:
        prompt = (
            f"Below is an instruction that describes a task. "
            f"Write a response that appropriately completes the request.\n\n"
            f"### Instruction:\n{instruction}\n\n"
            f"### Response:\n"
        )

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    # Reset routing logging flags across all 12 independent transformer blocks
    for layer in model.transformer.h:
        if hasattr(layer.mlp, 'routing_log'):
            layer.mlp.routing_log = []
            layer.mlp.track_routing = True

    with torch.no_grad():
        output_tokens = model.generate(
            **inputs,
            max_new_tokens=60,
            do_sample=True,
            top_k=40,
            top_p=0.9,
            temperature=0.6,
            pad_token_id=tokenizer.eos_token_id
        )

    for layer in model.transformer.h:
        if hasattr(layer.mlp, 'track_routing'):
            layer.mlp.track_routing = False

    response_text = tokenizer.decode(output_tokens[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

    # --- PRINT OUT VISUALIZATION RESULTS ---
    print(f"\nPROMPT: {instruction}")
    if input_context: print(f"INPUT:  {input_context}")
    print(f"RESPONSE:\n{response_text}\n")

    print("==================== MOE ROUTING MATRIX ====================")
    # The columns treat each layer's experts as structurally independent entities
    print(f"{'Layer':<6} | {'Expert 0':<12} | {'Expert 1':<12} | {'Expert 2':<12} | {'Expert 3':<12}")
    print("-" * 63)

    for layer_idx, layer in enumerate(model.transformer.h):
        if hasattr(layer.mlp, 'routing_log') and layer.mlp.routing_log:
            layer_counts = Counter(layer.mlp.routing_log)
            total_tokens = sum(layer_counts.values())

            # Safely calculate the unique routing percentage per expert per layer
            p0 = f"{layer_counts[0]/total_tokens*100:.1f}%" if total_tokens > 0 else "0.0%"
            p1 = f"{layer_counts[1]/total_tokens*100:.1f}%" if total_tokens > 0 else "0.0%"
            p2 = f"{layer_counts[2]/total_tokens*100:.1f}%" if total_tokens > 0 else "0.0%"
            p3 = f"{layer_counts[3]/total_tokens*100:.1f}%" if total_tokens > 0 else "0.0%"

            print(f"L{layer_idx:02d}   | {p0:<12} | {p1:<12} | {p2:<12} | {p3:<12}")

    print("===============================================================\n")

In [ ]:
model.to(device)
model.eval()

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

Test cases

In [ ]:
# a few test cases

# creative / list generation
evaluate_phrase_with_routing("List three healthy snacks.")

# context processing (instruction + input)
evaluate_phrase_with_routing(
    instruction="Correct the grammar in the sentence.",
    input_context="He do not have no money."
)

# close-ended logic
evaluate_phrase_with_routing("Is the sun a planet or a star?")


PROMPT: List three healthy snacks.
RESPONSE:
{ "name": "Food", "type": "food", "size": "5", "description": "Food", "type": "food", "size": "5", "description": "Size", "type": "food", "size": "5", "description": "

==================== MOE ROUTING MATRIX ====================
Layer  | Expert 0     | Expert 1     | Expert 2     | Expert 3    
---------------------------------------------------------------


PROMPT: Correct the grammar in the sentence.
INPUT:  He do not have no money.
RESPONSE:
He do not have no money.

### Request:

He do not have any money.

### Response:

He do not have any money.

### Request:

He do not have any money.

### Response:

He do not

==================== MOE ROUTING MATRIX ====================
Layer  | Expert 0     | Expert 1     | Expert 2     | Expert 3    
---------------------------------------------------------------


PROMPT: Is the sun a planet or a star?
RESPONSE:
The sun is a planet, or star, in the solar system. It is the closest thing to the su

In [ ]:
evaluate_phrase_with_routing("what is the capital of canada?")


PROMPT: what is the capital of canada?
RESPONSE:
Canada is the capital of the Spanish state of Canada.

==================== MOE ROUTING MATRIX ====================
Layer  | Expert 0     | Expert 1     | Expert 2     | Expert 3    
---------------------------------------------------------------
L00   | 0.0%         | 0.0%         | 2.0%         | 98.0%       
L01   | 0.0%         | 100.0%       | 0.0%         | 0.0%        
L02   | 0.0%         | 100.0%       | 0.0%         | 0.0%        
L03   | 0.0%         | 100.0%       | 0.0%         | 0.0%        
L04   | 0.0%         | 0.0%         | 0.0%         | 100.0%      
L05   | 0.0%         | 100.0%       | 0.0%         | 0.0%        
L06   | 0.0%         | 100.0%       | 0.0%         | 0.0%        
L07   | 0.0%         | 0.0%         | 100.0%       | 0.0%        
L08   | 0.0%         | 0.0%         | 0.0%         | 100.0%      
L09   | 98.0%        | 0.0%         | 2.0%         | 0.0%        
L10   | 98.0%        | 2.0%         | 0.0% 